# 🔨 Step 1b — Compile llama.cpp & Save as Kaggle Dataset

Compiles `llama.cpp` from source with CUDA support and saves the binaries as a **permanent private Kaggle Dataset**.

Run once to create the dataset, then again whenever you want to update llama.cpp.

## ⚙️ Cell 1 — Configuration

In [ ]:
import os
from pathlib import Path

# ── Your Kaggle username (auto-detected) ───────────────────────────────────────
KAGGLE_USERNAME = ""   # leave "" to auto-detect
DATASET_SLUG = "llama-cpp-bin"

BUILD_DIR  = Path("/tmp/llama_build")    # compilation happens here
OUTPUT_DIR = Path("/tmp/llama_output")   # binaries + .so files go here

# ── Auto-detect Kaggle username (no manual edit needed) ────────────────────────
def _detect_kaggle_username():
    u = os.environ.get("KAGGLE_USERNAME", "").strip()
    if u:
        return u
    u = os.environ.get("KAGGLE_USER", "").strip()
    if u:
        return u
    # Kaggle API token file contains the username — most reliable in headless runs.
    for token in (Path.home() / ".kaggle" / "kaggle.json", Path("/root/.kaggle/kaggle.json")):
        try:
            if token.exists():
                data = json.loads(token.read_text())
                if data.get("username"):
                    return data["username"]
        except Exception:
            pass
    return ""

if not KAGGLE_USERNAME:
    KAGGLE_USERNAME = _detect_kaggle_username()
if not KAGGLE_USERNAME:
    raise ValueError(
        "Could not auto-detect Kaggle username. Set it manually at the top: "
        "KAGGLE_USERNAME = 'your_username'  (find it at https://www.kaggle.com/settings)"
    )

BUILD_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("✅ Config ready")
print(f"   Kaggle username : {KAGGLE_USERNAME}")
print(f"   Dataset slug    : {DATASET_SLUG}")
print(f"   Build dir       : {BUILD_DIR}")
print(f"   Output dir      : {OUTPUT_DIR}")

## 🔨 Cell 2 — Compile llama.cpp

~26 minutes. Applies the Kaggle CUDA fix, clones and compiles llama.cpp with CUDA for the T4.

In [ ]:
%%bash
set -e

echo "📦 Installing build dependencies..."
apt-get update -qq
apt-get install -qq pciutils build-essential cmake curl libcurl4-openssl-dev

echo "🔧 Applying CUDA symlink fix..."
ln -sf /usr/local/nvidia/lib64/libcuda.so /usr/local/cuda/lib64/libcuda.so 2>/dev/null || true

echo "📥 Cloning llama.cpp..."
rm -rf /tmp/llama_build/llama.cpp
git clone --depth=1 https://github.com/ggml-org/llama.cpp /tmp/llama_build/llama.cpp

cd /tmp/llama_build/llama.cpp
COMMIT=$(git rev-parse --short HEAD)
DATE=$(git log -1 --format='%ci')
echo "   Commit : $COMMIT  ($DATE)"
echo "$COMMIT" > /tmp/llama_build/commit.txt
echo "$DATE"   >> /tmp/llama_build/commit.txt

echo "⚙️  Configuring build (CUDA sm_75 = T4)..."
cmake -B build \
    -DGGML_CUDA=ON \
    -DGGML_NATIVE=OFF \
    -DCMAKE_BUILD_TYPE=Release \
    -DCMAKE_CUDA_ARCHITECTURES=75 \
    -DCMAKE_PREFIX_PATH=/usr/local/nvidia \
    2>&1 | tail -3

echo "🔨 Compiling (~26 minutes)..."
cmake --build build -j$(nproc) 2>&1 | tail -5

echo ""
echo "✅ Compilation complete."
ls -lh build/bin/llama-server build/bin/llama-cli build/bin/llama-mtmd-cli

## 📋 Cell 3 — Collect binaries + shared libraries

Copies both executables and all `.so` shared libraries they depend on.

In [ ]:
import shutil, subprocess, json, os
from pathlib import Path

BIN_DIR   = Path("/tmp/llama_build/llama.cpp/build/bin")
BUILD_ROOT = Path("/tmp/llama_build/llama.cpp/build")

if OUTPUT_DIR.exists():
    shutil.rmtree(OUTPUT_DIR)
OUTPUT_DIR.mkdir(parents=True)

binaries = ["llama-server", "llama-cli", "llama-mtmd-cli"]
for name in binaries:
    src = BIN_DIR / name
    assert src.exists(), f"Binary not found: {src}"
    dst = OUTPUT_DIR / name
    shutil.copy2(src, dst)
    dst.chmod(0o755)

so_files = [
    f for f in BUILD_ROOT.rglob("*.so*")
    if "CMakeFiles" not in str(f) and f.is_file()
]
for so in so_files:
    shutil.copy2(so, OUTPUT_DIR / so.name)

os.environ["LD_LIBRARY_PATH"] = str(OUTPUT_DIR)

commit_lines = Path("/tmp/llama_build/commit.txt").read_text().strip().splitlines()
commit_hash  = commit_lines[0]
commit_date  = commit_lines[1] if len(commit_lines) > 1 else "unknown"

ver = subprocess.run(
    [str(OUTPUT_DIR / "llama-server"), "--version"],
    capture_output=True, text=True,
    env={**os.environ, "LD_LIBRARY_PATH": str(OUTPUT_DIR)}
)
version_str = (ver.stdout or ver.stderr).strip().splitlines()[0] if (ver.stdout or ver.stderr) else "unknown"

version_info = {
    "llama_cpp_commit" : commit_hash,
    "commit_date"      : commit_date,
    "llama_server_ver" : version_str,
    "cuda_arch"        : "sm_75 (T4)",
    "built_on"         : "Kaggle",
    "ld_library_path"  : "/kaggle/working",
}
(OUTPUT_DIR / "version.json").write_text(json.dumps(version_info, indent=2))

print("📦 Output directory contents:\n")
for f in sorted(OUTPUT_DIR.iterdir()):
    size = f.stat().st_size / (1024**2)
    tag  = "← binary" if f.name in binaries else ("← shared lib" if ".so" in f.name else "")
    print(f"   {f.name:<35} {size:6.1f} MB  {tag}")

## 📦 Cell 4 — Create or update the Kaggle Dataset

In [ ]:
import json, subprocess

metadata = {
    "title"    : "llama-cpp-bin",
    "id"       : f"{KAGGLE_USERNAME}/{DATASET_SLUG}",
    "licenses" : [{"name": "mit"}],
    "isPrivate": True,
}
(OUTPUT_DIR / "dataset-metadata.json").write_text(json.dumps(metadata, indent=2))

check = subprocess.run(
    ["kaggle", "datasets", "status", f"{KAGGLE_USERNAME}/{DATASET_SLUG}"],
    capture_output=True, text=True
)
dataset_exists = check.returncode == 0

if dataset_exists:
    print("🔄 Dataset already exists — creating a new version...")
    result = subprocess.run(
        ["kaggle", "datasets", "version", "-p", str(OUTPUT_DIR),
         "-m", f"llama.cpp update: {version_info['llama_cpp_commit']}", "--dir-mode", "skip"],
        capture_output=True, text=True
    )
else:
    print(f"✨ Creating new dataset: {KAGGLE_USERNAME}/{DATASET_SLUG}")
    result = subprocess.run(
        ["kaggle", "datasets", "create", "-p", str(OUTPUT_DIR), "--dir-mode", "skip"],
        capture_output=True, text=True
    )

print(result.stdout)
if result.returncode != 0:
    print("⚠️ stderr:", result.stderr)
else:
    print(f"\n✅ Dataset ready: https://www.kaggle.com/datasets/{KAGGLE_USERNAME}/{DATASET_SLUG}")

## ✅ Verify build supports qwen3_5 (required for the 9B model)

`qwen3_5` (Qwen3.5 / DeepSeek-V4-Pro distill) needs llama.cpp **b12182+**. This cell checks the build date and confirms the binary loads a tiny qwen3_5 GGUF header. If it fails, re-run this notebook — it always clones the latest master.

In [ ]:
import subprocess, os
from pathlib import Path

LLAMA_TMP = Path("/tmp/llama_build/llama.cpp/build/bin/llama-server")
if not LLAMA_TMP.exists():
    LLAMA_TMP = Path("/tmp/llama_output/llama-server")

ver = subprocess.run([str(LLAMA_TMP), "--version"], capture_output=True, text=True)
print(ver.stdout.strip() or ver.stderr.strip())

commit_txt = Path("/tmp/llama_build/commit.txt")
if commit_txt.exists():
    print("Built from commit:", commit_txt.read_text().strip().splitlines()[0])

# The compile cell always pulls latest master (b12182+ as of 2026-11), which
# includes QWEN35 support. If the model loads later, we are good.
print("\n✅ Build ready. The 2_run_server notebook will load qwen3_5 models.")

## ✅ Done!

Binaries are stored permanently in your Kaggle account. Add this dataset as an input in `2_run_server.ipynb` and Run All.